#### Copyright 2018 Google LLC.

In [0]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Systèmes de recommandation avec TensorFlow

Ce notebook Colab complète le cours sur les [systèmes de recommandation](https://developers.google.com/machine-learning/recommendation/). Plus précisément, nous allons utiliser la factorisation de matrice pour apprendre des embeddings d'utilisateurs et de films.

# Introduction

Nous allons créer un système de recommandation de films à partir du jeu de données [MovieLens](https://movielens.org/), disponible [ici](http://grouplens.org/datasets/movielens/). Les données sont des notes de films (sur une échelle de 1 à 5).

## Plan
  1. Exploration des données MovieLens (10 minutes)
  1. Préliminaires (25 minutes)
  1. Entraînement d'un modèle de factorisation de matrice (15 minutes)
  1. Inspection des embeddings (15 minutes)
  1. Régularisation de la factorisation de matrice (15 minutes)
  1. Entraînement d'un modèle softmax (30 minutes)

## Mise en place

Commençons par importer les paquets nécessaires.

In [0]:
# @title Imports (exécutez cette cellule)

# Installation des feuilles de calcul et import du module d'authentification.
USER_RATINGS = False
!pip install --upgrade -q gspread
from google.colab import auth
import gspread
from oauth2client.client import GoogleCredentials
from __future__ import print_function

import numpy as np
import pandas as pd
import collections
from mpl_toolkits.mplot3d import Axes3D
from IPython import display
from matplotlib import pyplot as plt
import sklearn
import sklearn.manifold
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
tf.logging.set_verbosity(tf.logging.ERROR)

# Ajout de quelques fonctions pratiques aux DataFrames Pandas.
pd.options.display.max_rows = 10
pd.options.display.float_format = '{:.3f}'.format
def mask(df, key, function):
  """Returns a filtered dataframe, by applying function to key"""
  return df[function(df[key])]

def flatten_cols(df):
  df.columns = [' '.join(col).strip() for col in df.columns.values]
  return df

pd.DataFrame.mask = mask
pd.DataFrame.flatten_cols = flatten_cols

# Installation d'Altair et activation de son rendu Colab.
print("Installation d'Altair...")
!pip install git+git://github.com/altair-viz/altair.git
import altair as alt
alt.data_transformers.enable('default', max_rows=None)
alt.renderers.enable('colab')
print("Installation d'Altair terminée.")



Nous téléchargeons ensuite les données MovieLens, et créons des DataFrames contenant les films, les utilisateurs et les notes.

In [0]:
# @title Chargement des données MovieLens (exécutez cette cellule).

# Téléchargement des données MovieLens.
print("Téléchargement des données movielens...")
from urllib.request import urlretrieve
import zipfile

urlretrieve("http://files.grouplens.org/datasets/movielens/ml-100k.zip", "movielens.zip")
zip_ref = zipfile.ZipFile('movielens.zip', "r")
zip_ref.extractall()
print("Terminé. Le jeu de données contient :")
print(zip_ref.read('ml-100k/u.info'))

# Chargement de chaque jeu de données (utilisateurs, films et notes).
users_cols = ['user_id', 'age', 'sex', 'occupation', 'zip_code']
users = pd.read_csv(
    'ml-100k/u.user', sep='|', names=users_cols, encoding='latin-1')

ratings_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
ratings = pd.read_csv(
    'ml-100k/u.data', sep='\t', names=ratings_cols, encoding='latin-1')

# Le fichier des films contient une caractéristique binaire par genre.
genre_cols = [
    "genre_unknown", "Action", "Adventure", "Animation", "Children", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]
movies_cols = [
    'movie_id', 'title', 'release_date', "video_release_date", "imdb_url"
] + genre_cols
movies = pd.read_csv(
    'ml-100k/u.item', sep='|', names=movies_cols, encoding='latin-1')

# Les identifiants commençant à 1, on les décale pour qu'ils commencent à 0.
users["user_id"] = users["user_id"].apply(lambda x: str(x-1))
movies["movie_id"] = movies["movie_id"].apply(lambda x: str(x-1))
movies["year"] = movies['release_date'].apply(lambda x: str(x).split('-')[-1])
ratings["movie_id"] = ratings["movie_id"].apply(lambda x: str(x-1))
ratings["user_id"] = ratings["user_id"].apply(lambda x: str(x-1))
ratings["rating"] = ratings["rating"].apply(lambda x: float(x))

# Calcul du nombre de films auxquels chaque genre est attribué.
genre_occurences = movies[genre_cols].sum().to_dict()

# Certains films pouvant appartenir à plusieurs genres, on crée différentes
# colonnes 'genre' comme suit :
# - all_genres : tous les genres actifs du film.
# - genre : tiré au hasard parmi les genres actifs.
def mark_genres(movies, genres):
  def get_random_genre(gs):
    active = [genre for genre, g in zip(genres, gs) if g==1]
    if len(active) == 0:
      return 'Other'
    return np.random.choice(active)
  def get_all_genres(gs):
    active = [genre for genre, g in zip(genres, gs) if g==1]
    if len(active) == 0:
      return 'Other'
    return '-'.join(active)
  movies['genre'] = [
      get_random_genre(gs) for gs in zip(*[movies[genre] for genre in genres])]
  movies['all_genres'] = [
      get_all_genres(gs) for gs in zip(*[movies[genre] for genre in genres])]

mark_genres(movies, genre_cols)

# Création d'un DataFrame fusionné contenant toutes les données movielens.
movielens = ratings.merge(movies, on='movie_id').merge(users, on='user_id')

# Utilitaire pour séparer les données en ensembles d'entraînement et de test.
def split_dataframe(df, holdout_fraction=0.1):
  """Splits a DataFrame into training and test sets.
  Args:
    df: a dataframe.
    holdout_fraction: fraction of dataframe rows to use in the test set.
  Returns:
    train: dataframe for training
    test: dataframe for testing
  """
  test = df.sample(frac=holdout_fraction, replace=False)
  train = df[~df.index.isin(test.index)]
  return train, test

# I. Exploration des données MovieLens
Avant de nous plonger dans la construction de modèles, inspectons notre jeu de données MovieLens. Il est généralement utile de comprendre les statistiques du jeu de données.

### Utilisateurs
Commençons par afficher quelques statistiques de base sur les caractéristiques numériques des utilisateurs.

In [0]:
users.describe()

Nous pouvons aussi afficher quelques statistiques de base sur les caractéristiques catégorielles des utilisateurs

In [0]:
users.describe(include=[np.object])

Nous pouvons aussi créer des histogrammes pour mieux comprendre la distribution des utilisateurs. Nous utilisons Altair pour créer un graphique interactif.

In [0]:
# @title Code de visualisation Altair (exécutez cette cellule)
# Les fonctions suivantes servent à générer des graphiques Altair interactifs.
# Nous afficherons des histogrammes des données, découpées selon un attribut donné.

# Création des filtres qui serviront à découper les données.
occupation_filter = alt.selection_multi(fields=["occupation"])
occupation_chart = alt.Chart().mark_bar().encode(
    x="count()",
    y=alt.Y("occupation:N"),
    color=alt.condition(
        occupation_filter,
        alt.Color("occupation:N", scale=alt.Scale(scheme='category20')),
        alt.value("lightgray")),
).properties(width=300, height=300, selection=occupation_filter)

# Une fonction qui génère un histogramme des données filtrées.
def filtered_hist(field, label, filter):
  """Creates a layered chart of histograms.
  The first layer (light gray) contains the histogram of the full data, and the
  second contains the histogram of the filtered data.
  Args:
    field: the field for which to generate the histogram.
    label: String label of the histogram.
    filter: an alt.Selection object to be used to filter the data.
  """
  base = alt.Chart().mark_bar().encode(
      x=alt.X(field, bin=alt.Bin(maxbins=10), title=label),
      y="count()",
  ).properties(
      width=300,
  )
  return alt.layer(
      base.transform_filter(filter),
      base.encode(color=alt.value('lightgray'), opacity=alt.value(.7)),
  ).resolve_scale(y='independent')


Regardons ensuite la distribution des notes par utilisateur. Cliquer sur une profession dans le graphique de droite filtre les données sur cette profession. L'histogramme correspondant est affiché en bleu, superposé à l'histogramme de l'ensemble des données (en gris clair). Vous pouvez utiliser MAJ+clic pour sélectionner plusieurs sous-ensembles.

Qu'observez-vous, et comment cela pourrait-il affecter les recommandations ?

In [0]:
users_ratings = (
    ratings
    .groupby('user_id', as_index=False)
    .agg({'rating': ['count', 'mean']})
    .flatten_cols()
    .merge(users, on='user_id')
)

# Création d'un graphique pour le décompte et d'un autre pour la moyenne.
alt.hconcat(
    filtered_hist('rating count', '# notes / utilisateur', occupation_filter),
    filtered_hist('rating mean', 'note moyenne par utilisateur', occupation_filter),
    occupation_chart,
    data=users_ratings)

### Films

Il est aussi utile de regarder les informations sur les films et leurs notes.

In [0]:
movies_ratings = movies.merge(
    ratings
    .groupby('movie_id', as_index=False)
    .agg({'rating': ['count', 'mean']})
    .flatten_cols(),
    on='movie_id')

genre_filter = alt.selection_multi(fields=['genre'])
genre_chart = alt.Chart().mark_bar().encode(
    x="count()",
    y=alt.Y('genre'),
    color=alt.condition(
        genre_filter,
        alt.Color("genre:N"),
        alt.value('lightgray'))
).properties(height=300, selection=genre_filter)

In [0]:
(movies_ratings[['title', 'rating count', 'rating mean']]
 .sort_values('rating count', ascending=False)
 .head(10))

In [0]:
(movies_ratings[['title', 'rating count', 'rating mean']]
 .mask('rating count', lambda x: x > 20)
 .sort_values('rating mean', ascending=False)
 .head(10))

Enfin, le dernier graphique montre la distribution du nombre de notes et de la note moyenne.

In [0]:
# Affichage du nombre de notes et de la note moyenne par film.
alt.hconcat(
    filtered_hist('rating count', '# notes / film', genre_filter),
    filtered_hist('rating mean', 'note moyenne par film', genre_filter),
    genre_chart,
    data=movies_ratings)

# II. Préliminaires

Notre objectif est de factoriser la matrice des notes $A$ en produit d'une matrice d'embeddings d'utilisateurs $U$ et d'une matrice d'embeddings de films $V$, telles que $A \approx UV^\top$ avec
$U = \begin{bmatrix} u_{1} \\ \hline \vdots \\ \hline u_{N} \end{bmatrix}$ et
$V = \begin{bmatrix} v_{1} \\ \hline \vdots \\ \hline v_{M} \end{bmatrix}$.

Ici
- $N$ est le nombre d'utilisateurs,
- $M$ est le nombre de films,
- $A_{ij}$ est la note du $j$-ième film par le $i$-ième utilisateur,
- chaque ligne $U_i$ est un vecteur (embedding) de dimension $d$ représentant l'utilisateur $i$,
- chaque ligne $V_j$ est un vecteur (embedding) de dimension $d$ représentant le film $j$,
- la prédiction du modèle pour la paire $(i, j)$ est le produit scalaire $\langle U_i, V_j \rangle$.

## Représentation creuse de la matrice des notes

La matrice des notes peut être très grande et, en général, la plupart de ses entrées ne sont pas observées, puisqu'un utilisateur donné ne note qu'un petit sous-ensemble des films. Pour une représentation efficace, nous utiliserons un [tf.SparseTensor](https://www.tensorflow.org/api_docs/python/tf/SparseTensor). Un `SparseTensor` représente la matrice à l'aide de trois tenseurs : `tf.SparseTensor(indices, values, dense_shape)` représente un tenseur où une valeur $A_{ij} = a$ est encodée en posant `indices[k] = [i, j]` et `values[k] = a`. Le dernier tenseur, `dense_shape`, précise la forme de la matrice complète sous-jacente.

#### Exemple jouet
Supposons que nous ayons $2$ utilisateurs et $4$ films. Notre DataFrame jouet contient trois notes :

user\_id | movie\_id | rating
--:|--:|--:
0 | 0 | 5.0
0 | 1 | 3.0
1 | 3 | 1.0

La matrice des notes correspondante est

$$
A =
\begin{bmatrix}
5.0 & 3.0 & 0 & 0 \\
0   &   0 & 0 & 1.0
\end{bmatrix}
$$

Et la représentation en SparseTensor est :
```python
SparseTensor(
  indices=[[0, 0], [0, 1], [1,3]],
  values=[5.0, 3.0, 1.0],
  dense_shape=[2, 4])
```

### Exercice 1 : construire une représentation tf.SparseTensor de la matrice des notes

Dans cet exercice, nous allons écrire une fonction qui transforme notre DataFrame `ratings` en `tf.SparseTensor`.

Indice : vous pouvez sélectionner les valeurs d'une colonne donnée d'un DataFrame `df` avec `df['column_name'].values`.

In [0]:
def build_rating_sparse_tensor(ratings_df):
  """
  Args:
    ratings_df: a pd.DataFrame with `user_id`, `movie_id` and `rating` columns.
  Returns:
    A tf.SparseTensor representing the ratings matrix.
  """
  # ========================= Complétez cette section ===========================
  # indices =
  # values =
  # ============================================================================

  return tf.SparseTensor(
      indices=indices,
      values=values,
      dense_shape=[users.shape[0], movies.shape[0]])

In [0]:
#@title Solution
def build_rating_sparse_tensor(ratings_df):
  """
  Args:
    ratings_df: a pd.DataFrame with `user_id`, `movie_id` and `rating` columns.
  Returns:
    a tf.SparseTensor representing the ratings matrix.
  """
  indices = ratings_df[['user_id', 'movie_id']].values
  values = ratings_df['rating'].values
  return tf.SparseTensor(
      indices=indices,
      values=values,
      dense_shape=[users.shape[0], movies.shape[0]])

## Calcul de l'erreur

Le modèle approxime la matrice des notes $A$ par un produit de rang faible $UV^\top$. Il nous faut un moyen de mesurer l'erreur d'approximation. Nous commencerons par utiliser l'erreur quadratique moyenne sur les entrées observées uniquement (nous y reviendrons plus tard). Elle est définie par

$$
\begin{align*}
\text{MSE}(A, UV^\top)
&= \frac{1}{|\Omega|}\sum_{(i, j) \in\Omega}{( A_{ij} - (UV^\top)_{ij})^2} \\
&= \frac{1}{|\Omega|}\sum_{(i, j) \in\Omega}{( A_{ij} - \langle U_i, V_j\rangle)^2}
\end{align*}
$$
où $\Omega$ est l'ensemble des notes observées, et $|\Omega|$ le cardinal de $\Omega$.

### Exercice 2 : erreur quadratique moyenne

Écrivez une fonction TensorFlow qui prend une matrice des notes creuse $A$ et les deux matrices d'embeddings $U, V$, et renvoie l'erreur quadratique moyenne $\text{MSE}(A, UV^\top)$.

Indices :
  * dans cette section, on ne considère que les entrées observées pour calculer la perte.
  * un `SparseTensor` `sp_x` est un tuple de trois tenseurs : `sp_x.indices`, `sp_x.values` et `sp_x.dense_shape`.
  * [`tf.gather_nd`](https://www.tensorflow.org/api_docs/python/tf/gather_nd) et [`tf.losses.mean_squared_error`](https://www.tensorflow.org/api_docs/python/tf/losses/mean_squared_error) pourront vous être utiles.

In [0]:
def sparse_mean_square_error(sparse_ratings, user_embeddings, movie_embeddings):
  """
  Args:
    sparse_ratings: A SparseTensor rating matrix, of dense_shape [N, M]
    user_embeddings: A dense Tensor U of shape [N, k] where k is the embedding
      dimension, such that U_i is the embedding of user i.
    movie_embeddings: A dense Tensor V of shape [M, k] where k is the embedding
      dimension, such that V_j is the embedding of movie j.
  Returns:
    A scalar Tensor representing the MSE between the true ratings and the
      model's predictions.
  """
  # ========================= Complétez cette section ===========================
  # loss =
  # ============================================================================
  return loss

In [0]:
#@title Solution
def sparse_mean_square_error(sparse_ratings, user_embeddings, movie_embeddings):
  """
  Args:
    sparse_ratings: A SparseTensor rating matrix, of dense_shape [N, M]
    user_embeddings: A dense Tensor U of shape [N, k] where k is the embedding
      dimension, such that U_i is the embedding of user i.
    movie_embeddings: A dense Tensor V of shape [M, k] where k is the embedding
      dimension, such that V_j is the embedding of movie j.
  Returns:
    A scalar Tensor representing the MSE between the true ratings and the
      model's predictions.
  """
  predictions = tf.gather_nd(
      tf.matmul(user_embeddings, movie_embeddings, transpose_b=True),
      sparse_ratings.indices)
  loss = tf.losses.mean_squared_error(sparse_ratings.values, predictions)
  return loss

Note : une approche consiste à calculer la matrice de prédiction complète $UV^\top$, puis à récupérer les entrées correspondant aux paires observées. Le coût mémoire de cette approche est en $O(NM)$. Pour le jeu de données MovieLens, cela ne pose pas de problème, car la matrice dense $N \times M$ est assez petite pour tenir en mémoire ($N = 943$, $M = 1682$).

Une autre approche (donnée dans la solution alternative ci-dessous) consiste à ne récupérer que les embeddings des paires observées, puis à calculer leurs produits scalaires. Le coût mémoire est en $O(|\Omega| d)$, où $d$ est la dimension des embeddings. Dans notre cas, $|\Omega| = 10^5$, et la dimension des embeddings est de l'ordre de $10$ : le coût mémoire des deux méthodes est donc comparable. Mais quand le nombre d'utilisateurs ou de films est bien plus grand, la première approche devient impraticable.

In [0]:
#@title Solution alternative
def sparse_mean_square_error(sparse_ratings, user_embeddings, movie_embeddings):
  """
  Args:
    sparse_ratings: A SparseTensor rating matrix, of dense_shape [N, M]
    user_embeddings: A dense Tensor U of shape [N, k] where k is the embedding
      dimension, such that U_i is the embedding of user i.
    movie_embeddings: A dense Tensor V of shape [M, k] where k is the embedding
      dimension, such that V_j is the embedding of movie j.
  Returns:
    A scalar Tensor representing the MSE between the true ratings and the
      model's predictions.
  """
  predictions = tf.reduce_sum(
      tf.gather(user_embeddings, sparse_ratings.indices[:, 0]) *
      tf.gather(movie_embeddings, sparse_ratings.indices[:, 1]),
      axis=1)
  loss = tf.losses.mean_squared_error(sparse_ratings.values, predictions)
  return loss

### Exercice 3 (optionnel) : ajouter vos propres notes au jeu de données

Vous pouvez ajouter vos propres notes au jeu de données. Si vous le faites, vous pourrez voir des recommandations pour vous-même.

Commencez par cocher la case ci-dessous. L'exécution de la cellule suivante vous authentifiera sur votre compte Google Drive et créera une feuille de calcul contenant tous les titres de films dans la colonne 'A'. Suivez le lien vers la feuille de calcul et prenez 3 minutes pour noter quelques films. Vos notes doivent être saisies dans la colonne 'B'.

In [0]:
USER_RATINGS = True #@param {type:"boolean"}

In [0]:
# @title Exécutez pour créer une feuille de calcul, puis saisissez-y vos notes.
# Authentification de l'utilisateur.
if USER_RATINGS:
  auth.authenticate_user()
  gc = gspread.authorize(GoogleCredentials.get_application_default())
  # Création de la feuille de calcul et affichage d'un lien vers elle.
  try:
    sh = gc.open('MovieLens-test')
  except(gspread.SpreadsheetNotFound):
    sh = gc.create('MovieLens-test')

  worksheet = sh.sheet1
  titles = movies['title'].values
  cell_list = worksheet.range(1, 1, len(titles), 1)
  for cell, title in zip(cell_list, titles):
    cell.value = title
  worksheet.update_cells(cell_list)
  print("Lien vers la feuille de calcul : "
        "https://docs.google.com/spreadsheets/d/{}/edit".format(sh.id))

Exécutez la cellule suivante pour charger vos notes et les ajouter au DataFrame principal `ratings`.

In [0]:
# @title Exécutez pour charger vos notes.
# Chargement des notes depuis la feuille de calcul et création d'un DataFrame.
if USER_RATINGS:
  my_ratings = pd.DataFrame.from_records(worksheet.get_all_values()).reset_index()
  my_ratings = my_ratings[my_ratings[1] != '']
  my_ratings = pd.DataFrame({
      'user_id': "943",
      'movie_id': list(map(str, my_ratings['index'])),
      'rating': list(map(float, my_ratings[1])),
  })
  # Suppression des notes précédentes.
  ratings = ratings[ratings.user_id != "943"]
  # Ajout des nouvelles notes.
  ratings = ratings.append(my_ratings, ignore_index=True)
  # Ajout du nouvel utilisateur au DataFrame des utilisateurs.
  if users.shape[0] == 943:
    users = users.append(users.iloc[942], ignore_index=True)
    users["user_id"][943] = "943"
  print("Vos %d notes ont été ajoutées ; vous avez très bon goût !" % len(my_ratings))
  ratings[ratings.user_id=="943"].merge(movies[['movie_id', 'title']])

# III. Entraînement d'un modèle de factorisation de matrice

## Classe utilitaire CFModel (Collaborative Filtering Model)
Il s'agit d'une classe simple pour entraîner un modèle de factorisation de matrice par descente de gradient stochastique.

Le constructeur de la classe prend
- les embeddings d'utilisateurs U (une `tf.Variable`).
- les embeddings de films V (une `tf.Variable`).
- une perte à optimiser (un `tf.Tensor`).
- une liste optionnelle de dictionnaires de métriques, chacun associant une chaîne (le nom de la métrique) à un tenseur. Ils sont évalués et affichés pendant l'entraînement (par exemple l'erreur d'entraînement et l'erreur de test).

Après l'entraînement, on peut accéder aux embeddings appris via le dictionnaire `model.embeddings`.

Exemple d'utilisation :
```
U_var = ...
V_var = ...
loss = ...
model = CFModel(U_var, V_var, loss)
model.train(iterations=100, learning_rate=1.0)
user_embeddings = model.embeddings['user_id']
movie_embeddings = model.embeddings['movie_id']
```

In [0]:
# @title Classe utilitaire CFModel (exécutez cette cellule)
class CFModel(object):
  """Simple class that represents a collaborative filtering model"""
  def __init__(self, embedding_vars, loss, metrics=None):
    """Initializes a CFModel.
    Args:
      embedding_vars: A dictionary of tf.Variables.
      loss: A float Tensor. The loss to optimize.
      metrics: optional list of dictionaries of Tensors. The metrics in each
        dictionary will be plotted in a separate figure during training.
    """
    self._embedding_vars = embedding_vars
    self._loss = loss
    self._metrics = metrics
    self._embeddings = {k: None for k in embedding_vars}
    self._session = None

  @property
  def embeddings(self):
    """The embeddings dictionary."""
    return self._embeddings

  def train(self, num_iterations=100, learning_rate=1.0, plot_results=True,
            optimizer=tf.train.GradientDescentOptimizer):
    """Trains the model.
    Args:
      iterations: number of iterations to run.
      learning_rate: optimizer learning rate.
      plot_results: whether to plot the results at the end of training.
      optimizer: the optimizer to use. Default to GradientDescentOptimizer.
    Returns:
      The metrics dictionary evaluated at the last iteration.
    """
    with self._loss.graph.as_default():
      opt = optimizer(learning_rate)
      train_op = opt.minimize(self._loss)
      local_init_op = tf.group(
          tf.variables_initializer(opt.variables()),
          tf.local_variables_initializer())
      if self._session is None:
        self._session = tf.Session()
        with self._session.as_default():
          self._session.run(tf.global_variables_initializer())
          self._session.run(tf.tables_initializer())
          tf.train.start_queue_runners()

    with self._session.as_default():
      local_init_op.run()
      iterations = []
      metrics = self._metrics or ({},)
      metrics_vals = [collections.defaultdict(list) for _ in self._metrics]

      # Entraînement et ajout des résultats.
      for i in range(num_iterations + 1):
        _, results = self._session.run((train_op, metrics))
        if (i % 10 == 0) or i == num_iterations:
          print("\r iteration %d: " % i + ", ".join(
                ["%s=%f" % (k, v) for r in results for k, v in r.items()]),
                end='')
          iterations.append(i)
          for metric_val, result in zip(metrics_vals, results):
            for k, v in result.items():
              metric_val[k].append(v)

      for k, v in self._embedding_vars.items():
        self._embeddings[k] = v.eval()

      if plot_results:
        # Affichage des métriques.
        num_subplots = len(metrics)+1
        fig = plt.figure()
        fig.set_size_inches(num_subplots*10, 8)
        for i, metric_vals in enumerate(metrics_vals):
          ax = fig.add_subplot(1, num_subplots, i+1)
          for k, v in metric_vals.items():
            ax.plot(iterations, v, label=k)
          ax.set_xlim([1, num_iterations])
          ax.legend()
      return results

### Exercice 4 : construire un modèle de factorisation de matrice et l'entraîner

À l'aide de votre fonction `sparse_mean_square_error`, écrivez une fonction qui construit un `CFModel` en créant les variables d'embeddings et les pertes d'entraînement et de test.

In [0]:
def build_model(ratings, embedding_dim=3, init_stddev=1.):
  """
  Args:
    ratings: a DataFrame of the ratings
    embedding_dim: the dimension of the embedding vectors.
    init_stddev: float, the standard deviation of the random initial embeddings.
  Returns:
    model: a CFModel.
  """
  # Séparation du DataFrame des notes en entraînement et test.
  train_ratings, test_ratings = split_dataframe(ratings)
  # Représentation en SparseTensor des jeux d'entraînement et de test.
  # ========================= Complétez cette section ===========================
  # A_train =
  # A_test =
  # ============================================================================
  # Initialisation des embeddings selon une loi normale.
  U = tf.Variable(tf.random_normal(
      [A_train.dense_shape[0], embedding_dim], stddev=init_stddev))
  V = tf.Variable(tf.random_normal(
      [A_train.dense_shape[1], embedding_dim], stddev=init_stddev))
  # ========================= Complétez cette section ===========================
  # train_loss =
  # test_loss =
  # ============================================================================
  metrics = {
      'train_error': train_loss,
      'test_error': test_loss
  }
  embeddings = {
      "user_id": U,
      "movie_id": V
  }
  return CFModel(embeddings, train_loss, [metrics])

In [0]:
#@title Solution
def build_model(ratings, embedding_dim=3, init_stddev=1.):
  """
  Args:
    ratings: a DataFrame of the ratings
    embedding_dim: the dimension of the embedding vectors.
    init_stddev: float, the standard deviation of the random initial embeddings.
  Returns:
    model: a CFModel.
  """
  # Séparation du DataFrame des notes en entraînement et test.
  train_ratings, test_ratings = split_dataframe(ratings)
  # Représentation en SparseTensor des jeux d'entraînement et de test.
  A_train = build_rating_sparse_tensor(train_ratings)
  A_test = build_rating_sparse_tensor(test_ratings)
  # Initialisation des embeddings selon une loi normale.
  U = tf.Variable(tf.random_normal(
      [A_train.dense_shape[0], embedding_dim], stddev=init_stddev))
  V = tf.Variable(tf.random_normal(
      [A_train.dense_shape[1], embedding_dim], stddev=init_stddev))
  train_loss = sparse_mean_square_error(A_train, U, V)
  test_loss = sparse_mean_square_error(A_test, U, V)
  metrics = {
      'train_error': train_loss,
      'test_error': test_loss
  }
  embeddings = {
      "user_id": U,
      "movie_id": V
  }
  return CFModel(embeddings, train_loss, [metrics])

Parfait, il est maintenant temps d'entraîner le modèle !

Exécutez la cellule suivante en essayant différents paramètres (dimension des embeddings, learning rate, itérations). Les erreurs d'entraînement et de test sont affichées à la fin de l'entraînement. Vous pouvez examiner ces valeurs pour valider les hyper-paramètres.

Note : en appelant à nouveau `model.train`, le modèle reprend l'entraînement à partir des valeurs actuelles des embeddings.

In [0]:
# Construction du modèle CF et entraînement.
model = build_model(ratings, embedding_dim=30, init_stddev=0.5)
model.train(num_iterations=1000, learning_rate=10.)

Les embeddings de films et d'utilisateurs sont aussi affichés dans la figure de droite. Quand la dimension des embeddings est supérieure à 3, ils sont projetés sur les 3 premières dimensions. La section suivante examinera les embeddings plus en détail.

# IV. Inspection des embeddings

Dans cette section, nous examinons de plus près les embeddings appris :
- en calculant vos recommandations,
- en regardant les plus proches voisins de quelques films,
- en regardant les normes des embeddings de films,
- en visualisant les embeddings dans un espace projeté.

### Exercice 5 : écrire une fonction qui calcule les scores des candidats
Commençons par écrire une fonction qui, étant donnés un embedding de requête $u \in \mathbb R^d$ et des embeddings d'éléments $V \in \mathbb R^{N \times d}$, calcule les scores des éléments.

Comme vu en cours, on peut utiliser différentes mesures de similarité, qui peuvent donner des résultats différents. Nous allons comparer les suivantes :
- produit scalaire : le score de l'élément j est $\langle u, V_j \rangle$.
- cosinus : le score de l'élément j est $\frac{\langle u, V_j \rangle}{\|u\|\|V_j\|}$.

Indices :
- vous pouvez utiliser [`np.dot`](https://docs.scipy.org/doc/numpy/reference/generated/numpy.dot.html) pour calculer le produit de deux np.Arrays.
- vous pouvez utiliser [`np.linalg.norm`](https://docs.scipy.org/doc/numpy-1.14.0/reference/generated/numpy.linalg.norm.html) pour calculer la norme d'un np.Array.

In [0]:
DOT = 'dot'
COSINE = 'cosine'
def compute_scores(query_embedding, item_embeddings, measure=DOT):
  """Computes the scores of the candidates given a query.
  Args:
    query_embedding: a vector of shape [k], representing the query embedding.
    item_embeddings: a matrix of shape [N, k], such that row i is the embedding
      of item i.
    measure: a string specifying the similarity measure to be used. Can be
      either DOT or COSINE.
  Returns:
    scores: a vector of shape [N], such that scores[i] is the score of item i.
  """
  # ========================= Complétez cette section ===========================
  # scores =
  # ============================================================================
  return scores

In [0]:
#@title Solution
DOT = 'dot'
COSINE = 'cosine'
def compute_scores(query_embedding, item_embeddings, measure=DOT):
  """Computes the scores of the candidates given a query.
  Args:
    query_embedding: a vector of shape [k], representing the query embedding.
    item_embeddings: a matrix of shape [N, k], such that row i is the embedding
      of item i.
    measure: a string specifying the similarity measure to be used. Can be
      either DOT or COSINE.
  Returns:
    scores: a vector of shape [N], such that scores[i] is the score of item i.
  """
  u = query_embedding
  V = item_embeddings
  if measure == COSINE:
    V = V / np.linalg.norm(V, axis=1, keepdims=True)
    u = u / np.linalg.norm(u)
  scores = u.dot(V.T)
  return scores

Munis de cette fonction, nous pouvons calculer des recommandations, où l'embedding de requête peut être celui d'un utilisateur ou celui d'un film.

In [0]:
# @title Recommandations aux utilisateurs et plus proches voisins (exécutez cette cellule)
def user_recommendations(model, measure=DOT, exclude_rated=False, k=6):
  if USER_RATINGS:
    scores = compute_scores(
        model.embeddings["user_id"][943], model.embeddings["movie_id"], measure)
    score_key = measure + ' score'
    df = pd.DataFrame({
        score_key: list(scores),
        'movie_id': movies['movie_id'],
        'titles': movies['title'],
        'genres': movies['all_genres'],
    })
    if exclude_rated:
      # suppression des films déjà notés
      rated_movies = ratings[ratings.user_id == "943"]["movie_id"].values
      df = df[df.movie_id.apply(lambda movie_id: movie_id not in rated_movies)]
    display.display(df.sort_values([score_key], ascending=False).head(k))  

def movie_neighbors(model, title_substring, measure=DOT, k=6):
  # Recherche des identifiants de films correspondant à la sous-chaîne donnée.
  ids =  movies[movies['title'].str.contains(title_substring)].index.values
  titles = movies.iloc[ids]['title'].values
  if len(titles) == 0:
    raise ValueError("Aucun film trouvé avec le titre %s" % title_substring)
  print("Plus proches voisins de : %s." % titles[0])
  if len(titles) > 1:
    print("[Plusieurs films correspondent. Autres candidats : {}]".format(
        ", ".join(titles[1:])))
  movie_id = ids[0]
  scores = compute_scores(
      model.embeddings["movie_id"][movie_id], model.embeddings["movie_id"],
      measure)
  score_key = measure + ' score'
  df = pd.DataFrame({
      score_key: list(scores),
      'titles': movies['title'],
      'genres': movies['all_genres']
  })
  display.display(df.sort_values([score_key], ascending=False).head(k))

### Vos recommandations

Si vous avez choisi de saisir vos notes, vous pouvez exécuter la cellule suivante pour générer des recommandations pour vous.

In [0]:
user_recommendations(model, measure=COSINE, k=5)

À quoi ressemblent les recommandations ?

### Plus proches voisins de films

Regardons les plus proches voisins de quelques films.

In [0]:
movie_neighbors(model, "Aladdin", DOT)
movie_neighbors(model, "Aladdin", COSINE)

Il semble que la qualité des embeddings appris ne soit pas très bonne. Nous y remédierons dans la section V en ajoutant plusieurs techniques de régularisation. Mais d'abord, examinons davantage les embeddings.

## Norme des embeddings de films

On peut aussi observer que les recommandations avec le produit scalaire et avec le cosinus diffèrent : avec le produit scalaire, le modèle a tendance à recommander des films populaires. Cela s'explique par le fait que, dans les modèles de factorisation de matrice, la norme de l'embedding est souvent corrélée à la popularité (les films populaires ont une norme plus grande), ce qui rend plus probable la recommandation d'éléments populaires. Nous pouvons confirmer cette hypothèse en triant les films par norme de leur embedding, comme dans la cellule suivante.

In [0]:
# @title Code de visualisation des embeddings (exécutez cette cellule)

def movie_embedding_norm(models):
  """Visualizes the norm and number of ratings of the movie embeddings.
  Args:
    model: A MFModel object.
  """
  if not isinstance(models, list):
    models = [models]
  df = pd.DataFrame({
      'title': movies['title'],
      'genre': movies['genre'],
      'num_ratings': movies_ratings['rating count'],
  })
  charts = []
  brush = alt.selection_interval()
  for i, model in enumerate(models):
    norm_key = 'norm'+str(i)
    df[norm_key] = np.linalg.norm(model.embeddings["movie_id"], axis=1)
    nearest = alt.selection(
        type='single', encodings=['x', 'y'], on='mouseover', nearest=True,
        empty='none')
    base = alt.Chart().mark_circle().encode(
        x='num_ratings',
        y=norm_key,
        color=alt.condition(brush, alt.value('#4c78a8'), alt.value('lightgray'))
    ).properties(
        selection=nearest).add_selection(brush)
    text = alt.Chart().mark_text(align='center', dx=5, dy=-5).encode(
        x='num_ratings', y=norm_key,
        text=alt.condition(nearest, 'title', alt.value('')))
    charts.append(alt.layer(base, text))
  return alt.hconcat(*charts, data=df)

def visualize_movie_embeddings(data, x, y):
  nearest = alt.selection(
      type='single', encodings=['x', 'y'], on='mouseover', nearest=True,
      empty='none')
  base = alt.Chart().mark_circle().encode(
      x=x,
      y=y,
      color=alt.condition(genre_filter, "genre", alt.value("whitesmoke")),
  ).properties(
      width=600,
      height=600,
      selection=nearest)
  text = alt.Chart().mark_text(align='left', dx=5, dy=-5).encode(
      x=x,
      y=y,
      text=alt.condition(nearest, 'title', alt.value('')))
  return alt.hconcat(alt.layer(base, text), genre_chart, data=data)

def tsne_movie_embeddings(model):
  """Visualizes the movie embeddings, projected using t-SNE with Cosine measure.
  Args:
    model: A MFModel object.
  """
  tsne = sklearn.manifold.TSNE(
      n_components=2, perplexity=40, metric='cosine', early_exaggeration=10.0,
      init='pca', verbose=True, n_iter=400)

  print('Exécution de t-SNE...')
  V_proj = tsne.fit_transform(model.embeddings["movie_id"])
  movies.loc[:,'x'] = V_proj[:, 0]
  movies.loc[:,'y'] = V_proj[:, 1]
  return visualize_movie_embeddings(movies, 'x', 'y')

In [0]:
movie_embedding_norm(model)

Note : selon l'initialisation du modèle, vous pourrez observer que certains films de niche (avec peu de notes) ont une norme élevée, ce qui produit des recommandations aberrantes. Cela peut arriver si l'embedding de ce film est initialisé avec une norme élevée. Comme le film a peu de notes, il est rarement mis à jour et peut garder sa norme élevée. La régularisation atténuera ce problème.

Essayez de changer la valeur de l'hyper-paramètre `init_stddev`. Une grandeur utile : la norme attendue d'un vecteur de dimension $d$ dont les entrées suivent $\mathcal N(0, \sigma^2)$ vaut environ $\sigma \sqrt d$.

Comment cela affecte-t-il la distribution des normes des embeddings, et le classement des films de plus grande norme ?

In [0]:
#@title Solution
model_lowinit = build_model(ratings, embedding_dim=30, init_stddev=0.05)
model_lowinit.train(num_iterations=1000, learning_rate=10.)
movie_neighbors(model_lowinit, "Aladdin", DOT)
movie_neighbors(model_lowinit, "Aladdin", COSINE)
movie_embedding_norm([model, model_lowinit])

## Visualisation des embeddings
Comme il est difficile de visualiser des embeddings dans un espace de grande dimension (quand la dimension des embeddings $k > 3$), une approche consiste à les projeter dans un espace de plus petite dimension. t-SNE (*T-distributed Stochastic Neighbor Embedding*) est un algorithme qui projette les embeddings en essayant de préserver leurs distances deux à deux. Il peut être utile pour la visualisation, mais il faut l'utiliser avec précaution. Pour en savoir plus sur l'utilisation de t-SNE, voir [How to Use t-SNE Effectively](https://distill.pub/2016/misread-tsne/).

In [0]:
tsne_movie_embeddings(model_lowinit)

Vous pouvez mettre en évidence les embeddings d'un genre donné en cliquant sur le panneau des genres (MAJ+clic pour sélectionner plusieurs genres).

On observe que les embeddings ne semblent pas avoir de structure notable, et que les embeddings d'un genre donné sont répartis dans tout l'espace. Cela confirme la faible qualité des embeddings appris. L'une des principales raisons, que nous traiterons dans la section suivante, est que nous n'avons entraîné le modèle que sur les paires observées, et sans régularisation.

# V. Régularisation de la factorisation de matrice

Dans la section précédente, notre perte était définie comme l'erreur quadratique moyenne sur la partie observée de la matrice des notes. Comme vu en cours, cela peut poser problème car le modèle n'apprend pas à placer les embeddings des films non pertinents. Ce phénomène est appelé *folding*.

Nous allons ajouter des termes de régularisation qui traitent ce problème. Nous utiliserons deux types de régularisation :
- La régularisation des paramètres du modèle. C'est un terme de régularisation $\ell_2$ classique sur les matrices d'embeddings : $r(U, V) =  \frac{1}{N} \sum_i \|U_i\|^2 + \frac{1}{M}\sum_j \|V_j\|^2$.
- Un a priori global qui pousse la prédiction de toute paire vers zéro, appelé terme de *gravité* : $g(U, V) = \frac{1}{MN} \sum_{i = 1}^N \sum_{j = 1}^M \langle U_i, V_j \rangle^2$.

La perte totale est alors donnée par
$$
\frac{1}{|\Omega|}\sum_{(i, j) \in \Omega} (A_{ij} - \langle U_i, V_j\rangle)^2 + \lambda _r r(U, V) + \lambda_g g(U, V)
$$
où $\lambda_r$ et $\lambda_g$ sont deux coefficients de régularisation (hyper-paramètres).

### Exercice 6 : construire un modèle de factorisation de matrice régularisé et l'entraîner
Écrivez une fonction qui construit un modèle régularisé. Une fonction `gravity(U, V)` vous est fournie : elle calcule le terme de gravité à partir des deux matrices d'embeddings $U$ et $V$.

In [0]:
def gravity(U, V):
  """Creates a gravity loss given two embedding matrices."""
  return 1. / (U.shape[0].value*V.shape[0].value) * tf.reduce_sum(
      tf.matmul(U, U, transpose_a=True) * tf.matmul(V, V, transpose_a=True))

def build_regularized_model(
    ratings, embedding_dim=3, regularization_coeff=.1, gravity_coeff=1.,
    init_stddev=0.1):
  """
  Args:
    ratings: the DataFrame of movie ratings.
    embedding_dim: The dimension of the embedding space.
    regularization_coeff: The regularization coefficient lambda.
    gravity_coeff: The gravity regularization coefficient lambda_g.
  Returns:
    A CFModel object that uses a regularized loss.
  """
  # Séparation du DataFrame des notes en entraînement et test.
  train_ratings, test_ratings = split_dataframe(ratings)
  # Représentation en SparseTensor des jeux d'entraînement et de test.
  A_train = build_rating_sparse_tensor(train_ratings)
  A_test = build_rating_sparse_tensor(test_ratings)
  U = tf.Variable(tf.random_normal(
      [A_train.dense_shape[0], embedding_dim], stddev=init_stddev))
  V = tf.Variable(tf.random_normal(
      [A_train.dense_shape[1], embedding_dim], stddev=init_stddev))

  # ========================= Complétez cette section ===========================
  # error_train =
  # error_test =
  # gravity_loss =
  # regularization_loss =
  # ============================================================================
  total_loss = error_train + regularization_loss + gravity_loss
  losses = {
      'train_error': error_train,
      'test_error': error_test,
  }
  loss_components = {
      'observed_loss': error_train,
      'regularization_loss': regularization_loss,
      'gravity_loss': gravity_loss,
  }
  embeddings = {"user_id": U, "movie_id": V}

  return CFModel(embeddings, total_loss, [losses, loss_components])

In [0]:
# @title Solution
def gravity(U, V):
  """Creates a gravity loss given two embedding matrices."""
  return 1. / (U.shape[0].value*V.shape[0].value) * tf.reduce_sum(
      tf.matmul(U, U, transpose_a=True) * tf.matmul(V, V, transpose_a=True))

def build_regularized_model(
    ratings, embedding_dim=3, regularization_coeff=.1, gravity_coeff=1.,
    init_stddev=0.1):
  """
  Args:
    ratings: the DataFrame of movie ratings.
    embedding_dim: The dimension of the embedding space.
    regularization_coeff: The regularization coefficient lambda.
    gravity_coeff: The gravity regularization coefficient lambda_g.
  Returns:
    A CFModel object that uses a regularized loss.
  """
  # Séparation du DataFrame des notes en entraînement et test.
  train_ratings, test_ratings = split_dataframe(ratings)
  # Représentation en SparseTensor des jeux d'entraînement et de test.
  A_train = build_rating_sparse_tensor(train_ratings)
  A_test = build_rating_sparse_tensor(test_ratings)
  U = tf.Variable(tf.random_normal(
      [A_train.dense_shape[0], embedding_dim], stddev=init_stddev))
  V = tf.Variable(tf.random_normal(
      [A_train.dense_shape[1], embedding_dim], stddev=init_stddev))

  error_train = sparse_mean_square_error(A_train, U, V)
  error_test = sparse_mean_square_error(A_test, U, V)
  gravity_loss = gravity_coeff * gravity(U, V)
  regularization_loss = regularization_coeff * (
      tf.reduce_sum(U*U)/U.shape[0].value + tf.reduce_sum(V*V)/V.shape[0].value)
  total_loss = error_train + regularization_loss + gravity_loss
  losses = {
      'train_error_observed': error_train,
      'test_error_observed': error_test,
  }
  loss_components = {
      'observed_loss': error_train,
      'regularization_loss': regularization_loss,
      'gravity_loss': gravity_loss,
  }
  embeddings = {"user_id": U, "movie_id": V}

  return CFModel(embeddings, total_loss, [losses, loss_components])

Il est maintenant temps d'entraîner le modèle régularisé ! Vous pouvez essayer différentes valeurs des coefficients de régularisation, et différentes dimensions d'embeddings.

In [0]:
reg_model = build_regularized_model(
    ratings, regularization_coeff=0.1, gravity_coeff=1.0, embedding_dim=35,
    init_stddev=.05)
reg_model.train(num_iterations=2000, learning_rate=20.)

On observe que l'ajout des termes de régularisation augmente la MSE, sur l'entraînement comme sur le test. Cependant, comme nous allons le voir, la qualité des recommandations s'améliore. Cela met en évidence une tension entre l'ajustement aux données observées et la minimisation des termes de régularisation. L'ajustement aux données observées met souvent l'accent sur l'apprentissage de fortes similarités (entre éléments avec beaucoup d'interactions), mais une bonne représentation par embeddings demande aussi d'apprendre de faibles similarités (entre éléments avec peu ou pas d'interactions).

### Examen des résultats
Voyons si les résultats avec régularisation sont meilleurs.

In [0]:
user_recommendations(reg_model, DOT, exclude_rated=True, k=10)

Avec un peu de chance, ces recommandations sont meilleures. Vous pouvez changer la mesure de similarité de COSINE à DOT et observer l'effet sur les recommandations.

Comme le modèle a tendance à recommander les éléments que vous avez bien notés, vous pouvez exclure les éléments que vous avez notés, avec `exclude_rated=True`.

Dans les cellules suivantes, nous affichons les plus proches voisins, les normes des embeddings et la projection t-SNE des embeddings de films.

In [0]:
movie_neighbors(reg_model, "Aladdin", DOT)
movie_neighbors(reg_model, "Aladdin", COSINE)

Nous comparons ici les normes des embeddings de `model` et de `reg_model`. Sélectionner un sous-ensemble d'embeddings les met en évidence sur les deux graphiques à la fois.

In [0]:
movie_embedding_norm([model, model_lowinit, reg_model])

In [0]:
# Visualisation des embeddings
tsne_movie_embeddings(reg_model)

On devrait observer que les embeddings sont bien plus structurés que sans régularisation. Essayez de sélectionner différents genres et observez comment ils ont tendance à former des groupes (par exemple Horror, Animation et Children).

### Conclusion
Ceci conclut cette section sur les modèles de factorisation de matrice. Notez que si la taille du problème est ici assez petite pour permettre un entraînement efficace par SGD, de nombreux problèmes réels doivent être entraînés avec des algorithmes plus spécialisés, comme les moindres carrés alternés (voir [tf.contrib.factorization.WALSMatrixFactorization](https://www.tensorflow.org/api_docs/python/tf/contrib/factorization/WALSMatrixFactorization) pour une implémentation TF).

# VI. Modèle softmax

Dans cette section, nous allons entraîner un modèle softmax simple qui prédit si un utilisateur donné a noté un film.

**Note** : si vous suivez la version en autonomie du cours, lisez bien la partie du cours sur l'entraînement softmax avant de travailler sur cette partie.

Le modèle prendra en entrée un vecteur de caractéristiques $x$ représentant la liste des films que l'utilisateur a notés. Nous partons du DataFrame des notes, que nous regroupons par user_id.

In [0]:
rated_movies = (ratings[["user_id", "movie_id"]]
                .groupby("user_id", as_index=False)
                .aggregate(lambda x: list(x)))
rated_movies.head()

Nous créons ensuite une fonction qui génère un batch d'exemples, où chaque exemple contient les caractéristiques suivantes :
- movie_id : un tenseur de chaînes, les identifiants des films que l'utilisateur a notés.
- genre : un tenseur de chaînes, les genres de ces films.
- year : un tenseur de chaînes, l'année de sortie.

In [0]:
#@title Code de génération des batchs (exécutez cette cellule)
years_dict = {
    movie: year for movie, year in zip(movies["movie_id"], movies["year"])
}
genres_dict = {
    movie: genres.split('-')
    for movie, genres in zip(movies["movie_id"], movies["all_genres"])
}

def make_batch(ratings, batch_size):
  """Creates a batch of examples.
  Args:
    ratings: A DataFrame of ratings such that examples["movie_id"] is a list of
      movies rated by a user.
    batch_size: The batch size.
  """
  def pad(x, fill):
    return pd.DataFrame.from_dict(x).fillna(fill).values

  movie = []
  year = []
  genre = []
  label = []
  for movie_ids in ratings["movie_id"].values:
    movie.append(movie_ids)
    genre.append([x for movie_id in movie_ids for x in genres_dict[movie_id]])
    year.append([years_dict[movie_id] for movie_id in movie_ids])
    label.append([int(movie_id) for movie_id in movie_ids])
  features = {
      "movie_id": pad(movie, ""),
      "year": pad(year, ""),
      "genre": pad(genre, ""),
      "label": pad(label, -1)
  }
  batch = (
      tf.data.Dataset.from_tensor_slices(features)
      .shuffle(1000)
      .repeat()
      .batch(batch_size)
      .make_one_shot_iterator()
      .get_next())
  return batch

def select_random(x):
  """Selectes a random elements from each row of x."""
  def to_float(x):
    return tf.cast(x, tf.float32)
  def to_int(x):
    return tf.cast(x, tf.int64)
  batch_size = tf.shape(x)[0]
  rn = tf.range(batch_size)
  nnz = to_float(tf.count_nonzero(x >= 0, axis=1))
  rnd = tf.random_uniform([batch_size])
  ids = tf.stack([to_int(rn), to_int(nnz * rnd)], axis=1)
  return to_int(tf.gather_nd(x, ids))


### Fonction de perte
Rappelons que le modèle softmax transforme les caractéristiques d'entrée $x$ en un embedding d'utilisateur $\psi(x) \in \mathbb R^d$, où $d$ est la dimension des embeddings. Ce vecteur est ensuite multiplié par une matrice d'embeddings de films $V \in \mathbb R^{m \times d}$ (où $m$ est le nombre de films), et la sortie finale du modèle est le softmax du produit
$$
\hat p(x) = \text{softmax}(\psi(x) V^\top).
$$
Étant donné une étiquette cible $y$, si l'on note $p = 1_y$ l'encodage one-hot de cette étiquette, la perte est l'entropie croisée entre $\hat p(x)$ et $p$.

### Exercice 7 : écrire une fonction de perte pour le modèle softmax

Dans cet exercice, nous allons écrire une fonction qui prend des tenseurs représentant les embeddings d'utilisateurs $\psi(x)$, les embeddings de films $V$ et l'étiquette cible $y$, et renvoie la perte d'entropie croisée.

Indice : vous pouvez utiliser la fonction [`tf.nn.sparse_softmax_cross_entropy_with_logits`](https://www.tensorflow.org/api_docs/python/tf/nn/sparse_softmax_cross_entropy_with_logits), qui prend des `logits` en entrée, où `logits` désigne le produit $\psi(x) V^\top$.

In [0]:
def softmax_loss(user_embeddings, movie_embeddings, labels):
  """Returns the cross-entropy loss of the softmax model.
  Args:
    user_embeddings: A tensor of shape [batch_size, embedding_dim].
    movie_embeddings: A tensor of shape [num_movies, embedding_dim].
    labels: A sparse tensor of dense_shape [batch_size, 1], such that
      labels[i] is the target label for example i.
  Returns:
    The mean cross-entropy loss.
  """
  # ========================= Complétez cette section ===========================
  # logits =
  # loss =
  # ============================================================================
  return loss

In [0]:
# @title Solution
def softmax_loss(user_embeddings, movie_embeddings, labels):
  """Returns the cross-entropy loss of the softmax model.
  Args:
    user_embeddings: A tensor of shape [batch_size, embedding_dim].
    movie_embeddings: A tensor of shape [num_movies, embedding_dim].
    labels: A tensor of [batch_size], such that labels[i] is the target label
      for example i.
  Returns:
    The mean cross-entropy loss.
  """
  # Vérification que les embeddings ont des dimensions compatibles
  user_emb_dim = user_embeddings.shape[1].value
  movie_emb_dim = movie_embeddings.shape[1].value
  if user_emb_dim != movie_emb_dim:
    raise ValueError(
        "The user embedding dimension %d should match the movie embedding "
        "dimension % d" % (user_emb_dim, movie_emb_dim))

  logits = tf.matmul(user_embeddings, movie_embeddings, transpose_b=True)
  loss = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
      logits=logits, labels=labels))
  return loss

### Exercice 8 : construire un modèle softmax, l'entraîner et examiner ses embeddings

Nous sommes maintenant prêts à construire un CFModel softmax. Complétez la fonction `build_softmax_model` de la cellule suivante. L'architecture du modèle est définie dans la fonction `create_user_embeddings` et illustrée par la figure ci-dessous. Les embeddings d'entrée (movie_id, genre et year) sont concaténés pour former la couche d'entrée, suivie de couches cachées dont les dimensions sont données par l'argument `hidden_dims`. Enfin, la dernière couche cachée est multipliée par les embeddings de films pour obtenir la couche de logits. Comme étiquette cible, nous utiliserons un movie_id tiré au hasard dans la liste des films que l'utilisateur a notés.

![Modèle softmax](https://github.com/google/eng-edu/blob/master/ml/recommendation-systems/images/softmax-model.png?raw=true)

Complétez la fonction ci-dessous en créant les colonnes de caractéristiques et les colonnes d'embeddings, puis les tenseurs de perte pour l'entraînement et le test (avec la fonction `softmax_loss` de l'exercice précédent).

In [0]:
def build_softmax_model(rated_movies, embedding_cols, hidden_dims):
  """Builds a Softmax model for MovieLens.
  Args:
    rated_movies: DataFrame of traing examples.
    embedding_cols: A dictionary mapping feature names (string) to embedding
      column objects. This will be used in tf.feature_column.input_layer() to
      create the input layer.
    hidden_dims: int list of the dimensions of the hidden layers.
  Returns:
    A CFModel object.
  """
  def create_network(features):
    """Maps input features dictionary to user embeddings.
    Args:
      features: A dictionary of input string tensors.
    Returns:
      outputs: A tensor of shape [batch_size, embedding_dim].
    """
    # Création d'un embedding en sac de mots pour chaque caractéristique creuse.
    inputs = tf.feature_column.input_layer(features, embedding_cols)
    # Couches cachées.
    input_dim = inputs.shape[1].value
    for i, output_dim in enumerate(hidden_dims):
      w = tf.get_variable(
          "hidden%d_w_" % i, shape=[input_dim, output_dim],
          initializer=tf.truncated_normal_initializer(
              stddev=1./np.sqrt(output_dim))) / 10.
      outputs = tf.matmul(inputs, w)
      input_dim = output_dim
      inputs = outputs
    return outputs

  train_rated_movies, test_rated_movies = split_dataframe(rated_movies)
  train_batch = make_batch(train_rated_movies, 200)
  test_batch = make_batch(test_rated_movies, 100)

  with tf.variable_scope("model", reuse=False):
    # Entraînement
    train_user_embeddings = create_network(train_batch)
    train_labels = select_random(train_batch["label"])
  with tf.variable_scope("model", reuse=True):
    # Test
    test_user_embeddings = create_network(test_batch)
    test_labels = select_random(test_batch["label"])
    movie_embeddings = tf.get_variable(
        "input_layer/movie_id_embedding/embedding_weights")

  # ========================= Complétez cette section ===========================
  # train_loss =
  # test_loss =
  # test_precision_at_10 =
  # ============================================================================

  metrics = (
      {"train_loss": train_loss, "test_loss": test_loss},
      {"test_precision_at_10": test_precision_at_10}
  )
  embeddings = {"movie_id": movie_embeddings}
  return CFModel(embeddings, train_loss, metrics)

In [0]:
# @title Solution

def build_softmax_model(rated_movies, embedding_cols, hidden_dims):
  """Builds a Softmax model for MovieLens.
  Args:
    rated_movies: DataFrame of traing examples.
    embedding_cols: A dictionary mapping feature names (string) to embedding
      column objects. This will be used in tf.feature_column.input_layer() to
      create the input layer.
    hidden_dims: int list of the dimensions of the hidden layers.
  Returns:
    A CFModel object.
  """
  def create_network(features):
    """Maps input features dictionary to user embeddings.
    Args:
      features: A dictionary of input string tensors.
    Returns:
      outputs: A tensor of shape [batch_size, embedding_dim].
    """
    # Création d'un embedding en sac de mots pour chaque caractéristique creuse.
    inputs = tf.feature_column.input_layer(features, embedding_cols)
    # Couches cachées.
    input_dim = inputs.shape[1].value
    for i, output_dim in enumerate(hidden_dims):
      w = tf.get_variable(
          "hidden%d_w_" % i, shape=[input_dim, output_dim],
          initializer=tf.truncated_normal_initializer(
              stddev=1./np.sqrt(output_dim))) / 10.
      outputs = tf.matmul(inputs, w)
      input_dim = output_dim
      inputs = outputs
    return outputs

  train_rated_movies, test_rated_movies = split_dataframe(rated_movies)
  train_batch = make_batch(train_rated_movies, 200)
  test_batch = make_batch(test_rated_movies, 100)

  with tf.variable_scope("model", reuse=False):
    # Entraînement
    train_user_embeddings = create_network(train_batch)
    train_labels = select_random(train_batch["label"])
  with tf.variable_scope("model", reuse=True):
    # Test
    test_user_embeddings = create_network(test_batch)
    test_labels = select_random(test_batch["label"])
    movie_embeddings = tf.get_variable(
        "input_layer/movie_id_embedding/embedding_weights")

  test_loss = softmax_loss(
      test_user_embeddings, movie_embeddings, test_labels)
  train_loss = softmax_loss(
      train_user_embeddings, movie_embeddings, train_labels)
  _, test_precision_at_10 = tf.metrics.precision_at_k(
      labels=test_labels,
      predictions=tf.matmul(test_user_embeddings, movie_embeddings, transpose_b=True),
      k=10)

  metrics = (
      {"train_loss": train_loss, "test_loss": test_loss},
      {"test_precision_at_10": test_precision_at_10}
  )
  embeddings = {"movie_id": movie_embeddings}
  return CFModel(embeddings, train_loss, metrics)

### Entraînement du modèle softmax

Nous sommes maintenant prêts à entraîner le modèle softmax. Vous pouvez régler les hyper-paramètres suivants :
- le learning rate
- le nombre d'itérations. Note : vous pouvez exécuter à nouveau `softmax_model.train()` pour poursuivre l'entraînement du modèle à partir de son état actuel.
- les dimensions des embeddings d'entrée (l'argument `input_dims`)
- le nombre de couches cachées et la taille de chaque couche (l'argument `hidden_dims`)

Note : comme nos caractéristiques d'entrée sont des chaînes (movie_id, genre et year), il faut les transformer en identifiants entiers. C'est le rôle de [`tf.feature_column.categorical_column_with_vocabulary_list`](https://www.tensorflow.org/api_docs/python/tf/feature_column/categorical_column_with_vocabulary_list), qui prend une liste de vocabulaire donnant toutes les valeurs que la caractéristique peut prendre. Chaque identifiant est ensuite associé à un vecteur d'embedding avec [`tf.feature_column.embedding_column`](https://www.tensorflow.org/api_docs/python/tf/feature_column/embedding_column).

In [0]:
# Création des colonnes d'embeddings des caractéristiques
def make_embedding_col(key, embedding_dim):
  categorical_col = tf.feature_column.categorical_column_with_vocabulary_list(
      key=key, vocabulary_list=list(set(movies[key].values)), num_oov_buckets=0)
  return tf.feature_column.embedding_column(
      categorical_column=categorical_col, dimension=embedding_dim,
      # initialisation par défaut : loi normale tronquée d'écart-type 1/sqrt(dimension)
      combiner='mean')

with tf.Graph().as_default():
  softmax_model = build_softmax_model(
      rated_movies,
      embedding_cols=[
          make_embedding_col("movie_id", 35),
          make_embedding_col("genre", 3),
          make_embedding_col("year", 2),
      ],
      hidden_dims=[35])

softmax_model.train(
    learning_rate=8., num_iterations=3000, optimizer=tf.train.AdagradOptimizer)

### Examen des embeddings

Nous pouvons examiner les embeddings de films comme pour les modèles précédents. Notez que dans ce cas, les embeddings de films servent à la fois d'embeddings d'entrée (pour la représentation en sac de mots de l'historique de l'utilisateur) et de poids du softmax.

In [0]:
movie_neighbors(softmax_model, "Aladdin", DOT)
movie_neighbors(softmax_model, "Aladdin", COSINE)

In [0]:
movie_embedding_norm([reg_model, softmax_model])

In [0]:
tsne_movie_embeddings(softmax_model)

## Félicitations !

Vous avez terminé ce notebook Colab.

Si vous souhaitez explorer davantage ces modèles, nous vous encourageons à essayer différents hyper-paramètres et à observer leur effet sur la qualité du modèle et sur la structure de l'espace des embeddings. Quelques suggestions :
- Changez la dimension des embeddings.
- Dans le modèle softmax : changez le nombre de couches cachées et les caractéristiques d'entrée. Par exemple, vous pouvez essayer un modèle sans couche cachée, avec seulement les identifiants de films en entrée.
- Utilisez d'autres mesures de similarité : dans ce notebook Colab, nous avons utilisé le produit scalaire $d(u, V_j) = \langle u, V_j \rangle$ et le cosinus $d(u, V_j) = \frac{\langle u, V_j \rangle}{\|u\|\|V_j\|}$, et vu comment les normes des embeddings affectent les recommandations. Vous pouvez aussi essayer d'autres variantes qui appliquent une transformation à la norme, par exemple $d(u, V_j) = \frac{\langle u, V_j \rangle}{\|V_j\|^\alpha}$.